In [ ]:
#Titanic Data Analysis and JSON Export
##Student : Jay
##Task: Analyze Titanic passenger data, engineer features and export to JSON

import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path

#Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"

#Create data directory
DATA_DIR.mkdir(exist_ok=True)

print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data\titanic.csv


In [3]:
#Load Titanic dataset
df = pd.read_csv(CSV_FILE)

print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

In [5]:
#Select only columns with numbers
numeric_columns = df.select_dtypes(include=np.number).columns

print("DESCRIPTIVE STATISTICS")

for col in numeric_columns:
    mean_val = df[col].mean()
    median_val = df[col].median()
    std_val = df[col].std()
    print(f"\n{col}:")
    print(f"  Mean:   {mean_val:.2f}")
    print(f"  Median: {median_val:.2f}")
    print(f"  Std:    {std_val:.2f}")

DESCRIPTIVE STATISTICS

PassengerId:
  Mean:   446.00
  Median: 446.00
  Std:    257.35

Survived:
  Mean:   0.38
  Median: 0.00
  Std:    0.49

Pclass:
  Mean:   2.31
  Median: 3.00
  Std:    0.84

Age:
  Mean:   29.70
  Median: 28.00
  Std:    14.53

SibSp:
  Mean:   0.52
  Median: 0.00
  Std:    1.10

Parch:
  Mean:   0.38
  Median: 0.00
  Std:    0.81

Fare:
  Mean:   32.20
  Median: 14.45
  Std:    49.69


In [6]:
#Missing values

print("MISSING VALUES ANALYSIS")

missing_data = {}

for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    missing_data[col] = {'count': missing_count, 'percent': missing_percent}
    if missing_count > 0:
        print(f"\n{col}:")
        print(f"  Missing count:   {missing_count}")
        print(f"  Missing percent: {missing_percent:.2f}%")

MISSING VALUES ANALYSIS

Age:
  Missing count:   177
  Missing percent: 19.87%

Cabin:
  Missing count:   687
  Missing percent: 77.10%

Embarked:
  Missing count:   2
  Missing percent: 0.22%


In [10]:
#Feature engineering
df_features = df.copy()

#Feature - Family Size
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))

# Feature - Passenger is Alone
df_features['IsAlone'] = (df_features['FamilySize'] == 1).astype(int)
print(df_features[['FamilySize', 'IsAlone']].head(10))

#Feature - Age Grouping
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 30:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Senior'

df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
   FamilySize  IsAlone
0           2        0
1           2        0
2           1        1
3           2        0
4           1        1
5           1        1
6           1        1
7           5        0
8           3        0
9           2        0
    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0       Senior
7   2.0        Child
8  27.0  Young Adult
9  14.0        Child


In [14]:
#Survivors and non-survivors feature difference

print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")

print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)

# Statistical test: Do these features help differentiate?

print("FEATURE DIFFERENTIATION ANALYSIS")

survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]

print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

#Sense check with Alone passenger statistics
print("\nSurvival Rate by IsAlone:")
print(df_features.groupby('IsAlone')['Survived'].mean())

FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0  1.186076
FEATURE DIFFERENTIATION ANALYSIS

Family Size:
  Survived mean: 1.94
  Not Survived mean: 1.88
  Difference: 0.06

Survival Rate by IsAlone:
IsAlone
0    0.505650
1    0.303538
Name: Survived, dtype: float64


In [18]:
#Defining passenger class
class Passenger:
    
    def __init__(self, passenger_id, name, age, sex, survived, pclass, 
                 fare, embarked=None, family_size=None, is_alone=None):
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = int(is_alone) if pd.notna(is_alone) else None

    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone
        }

print("Passenger class defined with to_dict()!")

# Quick test: create one passenger manually and check it works
test_passenger = Passenger(
    passenger_id=1, name="Test Person", age=32, sex="female",
    survived=1, pclass=1, fare=50.0, embarked="S", family_size=2, is_alone=0
)
print(test_passenger.to_dict())

Passenger class defined with to_dict()!
{'passenger_id': 1, 'name': 'Test Person', 'age': 32.0, 'sex': 'female', 'survived': 1, 'pclass': 1, 'fare': 50.0, 'embarked': 'S', 'family_size': 2, 'is_alone': 0}


In [ ]:
#Creating Passenger Dataset
class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []  # Will store Passenger objects
        self._create_passengers()
    
    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        for idx, row in self.dataframe.iterrows():
            passenger = Passenger(
                passenger_id=row.get('PassengerId', idx),
                name=row.get('Name'),
                age=row.get('Age'),
                sex=row.get('Sex'),
                survived=row.get('Survived'),
                pclass=row.get('Pclass'),
                fare=row.get('Fare'),
                embarked=row.get('Embarked'),
                family_size=row.get('FamilySize'),
                is_alone=row.get('IsAlone')
            )
            self.passengers.append(passenger)

print("TitanicDataset class defined!")

# Quick test: build the dataset and check the count
dataset = TitanicDataset(df_features)
print(f"Total passengers created: {len(dataset.passengers)}")
print(f"First passenger: {dataset.passengers[0].to_dict()}")

TitanicDataset class defined!
Total passengers created: 891
First passenger: {'passenger_id': 1, 'name': 'Braund, Mr. Owen Harris', 'age': 22.0, 'sex': 'male', 'survived': 0, 'pclass': 3, 'fare': 7.25, 'embarked': 'S', 'family_size': 2, 'is_alone': 0}


In [ ]:
class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []
        self._create_passengers()
    
    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        for idx, row in self.dataframe.iterrows():
            passenger = Passenger(
                passenger_id=row.get('PassengerId', idx),
                name=row.get('Name'),
                age=row.get('Age'),
                sex=row.get('Sex'),
                survived=row.get('Survived'),
                pclass=row.get('Pclass'),
                fare=row.get('Fare'),
                embarked=row.get('Embarked'),
                family_size=row.get('FamilySize'),
                is_alone=row.get('IsAlone')
            )
            self.passengers.append(passenger)
    
    def to_json(self, filename='titanic_data.json'):
        """Export dataset to JSON file."""
        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                'total_passengers': len(self.passengers),
                'survival_rate': float(self.dataframe['Survived'].mean())
            },
            'passengers': [p.to_dict() for p in self.passengers]
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2)
        
        print(f"Data exported to {filename}")
        return data

print("TitanicDataset class defined with to_json()!")

# Build the dataset and export it
dataset = TitanicDataset(df_features)
exported_data = dataset.to_json(JSON_FILE)